# 01 Document Ingestion and Retrieval

## Project context

This notebook builds Phase 0 of the AI Financial Research Agent. The goal is to ingest local finance documents, split them into searchable chunks, retrieve relevant evidence for research questions, and keep outputs traceable to source text.

Phase 0 uses local synthetic sample documents only. No paid APIs, OpenAI API calls, embeddings, or memo generation are used.

## Ingestion objective

- Load local `.txt` sample finance documents.
- Save document-level metadata.
- Clean and chunk text.
- Build a TF-IDF retrieval index.
- Run sample finance research queries.
- Save retrieval results and evidence tables for Phase 1.

## Sample document description

The sample documents are synthetic portfolio examples. Themes include fintech lending growth, credit risk, inflation and interest-rate risk, digital payments, bank profitability, and market volatility.

In [ ]:
from pathlib import Path
import sys

import pandas as pd

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "src").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.append(str(PROJECT_ROOT))

from src.config import (
    SAMPLE_DOCS_DIR,
    PROCESSED_DATA_DIR,
    CHUNKS_DIR,
    RETRIEVAL_DIR,
    EVIDENCE_DIR,
    FIGURES_DIR,
)
from src.ingestion import load_document_corpus, save_corpus_metadata
from src.chunking import build_chunk_table, save_chunks, clean_text
from src.retrieval import (
    build_tfidf_index,
    search_chunks,
    build_evidence_table,
    save_retrieval_results,
)
from src.visualization import (
    plot_chunks_by_document,
    plot_top_terms_overall,
    plot_retrieval_score_distribution,
)

for directory in [PROCESSED_DATA_DIR, CHUNKS_DIR, RETRIEVAL_DIR, EVIDENCE_DIR, FIGURES_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

## Load sample documents

In [ ]:
corpus = load_document_corpus(SAMPLE_DOCS_DIR)
len(corpus), [doc["title"] for doc in corpus]

## Corpus metadata

In [ ]:
metadata = save_corpus_metadata(corpus, PROCESSED_DATA_DIR / "corpus_metadata.csv")
metadata

## Text cleaning

In [ ]:
cleaned_preview = clean_text(corpus[0]["text"])
cleaned_preview[:500]

## Chunking

In [ ]:
chunks = build_chunk_table(corpus)
save_chunks(chunks, CHUNKS_DIR / "document_chunks.csv")
chunks[["chunk_id", "title", "chunk_index", "word_count"]].head(), chunks.shape

## TF-IDF index creation

In [ ]:
vectorizer, matrix = build_tfidf_index(chunks)
matrix.shape

## Sample retrieval queries

In [ ]:
queries = [
    "What are the main credit risk concerns?",
    "How does inflation affect financial institutions?",
    "What risks affect fintech lending growth?",
    "What does the document set say about digital payments?",
    "What market risks affect portfolio performance?",
]

retrieval_frames = [
    search_chunks(query, vectorizer, matrix, chunks, top_k=5)
    for query in queries
]
retrieval_results = pd.concat(retrieval_frames, ignore_index=True)
save_retrieval_results(retrieval_results, RETRIEVAL_DIR / "sample_retrieval_results.csv")
retrieval_results[["query", "rank", "retrieval_score", "title", "chunk_id"]].head(10)

## Evidence table

In [ ]:
evidence_table = build_evidence_table(retrieval_results)
evidence_table.to_csv(EVIDENCE_DIR / "evidence_table.csv", index=False)
evidence_table.head(10)

## Figures

In [ ]:
fig = plot_chunks_by_document(chunks)
fig.savefig(FIGURES_DIR / "chunks_by_document.png", dpi=150, bbox_inches="tight")
fig

In [ ]:
fig = plot_top_terms_overall(chunks)
fig.savefig(FIGURES_DIR / "top_terms_overall.png", dpi=150, bbox_inches="tight")
fig

In [ ]:
fig = plot_retrieval_score_distribution(retrieval_results)
fig.savefig(FIGURES_DIR / "retrieval_score_distribution.png", dpi=150, bbox_inches="tight")
fig

## Initial interpretation

The retrieval pipeline can connect finance questions to source chunks without using a paid API. This creates a traceable evidence layer for later memo generation. The highest ranked chunks generally align with the query themes: credit risk queries retrieve borrower behavior and fintech lending notes; inflation queries retrieve rate and funding cost notes; digital payments queries retrieve the payments adoption note.

## Limitations

- Sample documents are synthetic and short.
- TF-IDF is lexical and does not understand meaning like an embedding model.
- No LLM generation or final research memo is built in Phase 0.
- No PDF parser, table parser, or production document ingestion layer yet.

## Next steps for Phase 1

- Add memo drafting from retrieved evidence.
- Add risk flag extraction.
- Add citation grounding and evaluation checks.
- Compare retrieval coverage across questions.